# The HALD Knowledge Graph

HALD is a knowledge graph for human aging and longevity. It is described at this webpage on [Figshare](https://figshare.com/articles/dataset/HALD_a_human_aging_and_longevity_knowledge_graph_for_precision_gerontology_and_geroscience_analyses/22828196). This include versions in JSON and CSV. The CSV versions are structured for a particular package which we will not be using, so we will use the json packages which are more general.

First we'll create a data directory.

In [1]:
import os
Download = False # After you have run this the first time, set this to False to avoid repeated downloads of the data
datadir = "HALD_Dataset"
if not os.path.exists(datadir):
    os.mkdir(datadir)

Now we define the list of the files that we want to download. We'll define a *list* of *tuples*, with each tuple representing one of the files that we want to fetch, specifying three things:

* The name that we want the file to be called.
* The URL from where it will be downloaded.
* The MD5 checksum which will allow us to verify the downloaded file's integrity.

In [2]:
# List of files to download
filelist = [
    ("Entity_info.json", "https://figshare.com/ndownloader/files/43612509", '1746cde24a1bac0460f1ccf646608cc9'),
    ("Literature_Info.json", "https://figshare.com/ndownloader/files/43612512", "10b78e8ec30f5b85f2a58d8fe24f056b"),
    ("Longevity_Biomarkers.json", "https://figshare.com/ndownloader/files/43612497", "0dbd9c3f8474dc3cd744ed38af460d75"),
    ("Relation_Info.json", "https://figshare.com/ndownloader/files/43612506", "0c1fa199269adc58f64ad4d5b9fd87b9"),
    ("Aging_Biomarkers.json", "https://figshare.com/ndownloader/files/43612503", "abd0eb6cb7295ae500c5d676b7797324")
]

Now we can download the files. For each file in `filelist` we will:

* Download the file from the URL.
* If the download request indicates that the download is unsuccessful, print an error.
* If the download is successfull, verify the checksum and if that is correct, write the file to disk in `datadir`

In [3]:
import requests
import hashlib

if Download:
    for f in filelist:
        response = requests.get(f[1])
        file_Path = datadir + "/" + f[0]
        if response.status_code != 200:
            print('Failed to download file {f[0]} from {f[1]}')
        else:
            m = hashlib.md5()
            m.update(response.content)
            if m.hexdigest() == f[2]:
                print(f"SUCCESS: File {f[0]} downloaded from {f[1]} with correct checksum {f[2]}")
                with open(file_Path, 'wb') as file:
                    file.write(response.content)
            else:
                print(f"ERROR: File {f[0]} downloaded from {f[1]} with incorrect checksum {m.hexdigest()} (should be {f[2]})")            


## What does the data look like?

Let us inspect these files. The two key files here are those containing the *entities* (nodes) and the *edges* (relations). The following code loads the contents of these files into two dictionaries.

In [4]:
import json

def load_json(fname):
    with open(fname, 'rb') as file:
        return json.load(file)

EntityInfo = load_json(f"{datadir}/{filelist[0][0]}")
RelationInfo = load_json(f"{datadir}/{filelist[3][0]}")


## 1.  Properties of the dataset

Examine the dataset to understand how it is structured, and answer the following questions:

* How many types of entity are there and what are their types?
* How many instances of each type of entity are there?
* How many different types of relation are there?
* What are the five most common relations and how many times is each one present?



In [5]:
# Let's take a look at an example to see what we should be looking for

print(EntityInfo.keys())
for i in EntityInfo['MLH1'][0]:
    print(f"{i}: {EntityInfo['MLH1'][0][i]}")

dict_keys(['MLH1', 'CD4', 'INS', 'MAPT', 'MYC', 'GSR', 'SOD2', 'CRP', 'IL6', 'SIRT1', 'CHGA', 'CFB', 'SKIV2L', 'TNXB', 'FKBPL', 'NOTCH4', 'CFH', 'HTRA1', 'GCG', 'IGF1', 'GH1', 'GHRH', 'WRN', 'NFKB1', 'SHBG', 'PIAS4', 'CCL2', 'RECQL4', 'BLM', 'ALB', 'TNF', 'BCAM', 'CD151', 'GGH', 'FGF23', 'PTH', 'JUNB', 'H2AZ1', 'PAPPA2', 'ELN', 'KIT', 'CSF2', 'VEGFA', 'MYO5A', 'MTOR', 'KLK3', 'AR', 'ACE', 'LMNB1', 'LMNA', 'NUP62', 'ULK1', 'MAP1LC3A', 'PIK3R2', 'IAPP', 'VDR', 'CLPS', 'APOD', 'FERMT2', 'MS4A6A', 'ABCA7', 'SORL1', 'HTT', 'APOB', 'RAF1', 'MAPK3', 'MAPK1', 'MAP2K1', 'MAP2K2', 'CFI', 'SERPINA1', 'IL7', 'KL', 'BECN1', 'NFE2L2', 'SENP7', 'MOB1B', 'CARMIL1', 'PRRC2A', 'TERF2', 'RFWD3', 'PARP1', 'POT1', 'ATM', 'MPHOSPH6', 'PPARGC1A', 'FNDC5', 'BDNF', 'NTRK2', 'CD8A', 'IFITM3', 'TRIM22', 'LY6E', 'IFNAR1', 'CTNNB1', 'APOL1', 'VWF', 'ATR', 'RNF8', 'BRCA1', 'TP53BP1', 'RETN', 'CXCL8', 'IL10', 'IL1B', 'IL13RA2', 'CXCR4', 'POU5F1', 'NANOG', 'IL2', 'APOE', 'NDRG2', 'BACE1', 'GGA3', 'CDK5', 'PIN1', 'STA

In [6]:
# The key things we need here is the "type" variable
EntityTypes = set()
for k in EntityInfo.keys():
    EntityTypes.add(EntityInfo[k][0]['type'])

print(f"There are {len(EntityInfo)} entities")
print(f"There are {len(EntityTypes)} types of entity")
print(EntityTypes)

There are 12257 entities
There are 10 types of entity
{'Protein', 'Lipid', 'RNA', 'Pharmaceutical Preparations', 'Gene', 'Mutation', 'Peptide', 'Disease', 'Carbohydrate', 'Toxin'}


In [7]:
# Now to look at the Relations
RelationKeys = list(RelationInfo.keys())
print(RelationKeys[0])


for k in RelationInfo['Pulmonary Disease, Chronic Obstructive-defined-Inflammation']:
    print(f"{k}: {RelationInfo['Pulmonary Disease, Chronic Obstructive-defined-Inflammation'][k]}")

Pulmonary Disease, Chronic Obstructive-defined-Inflammation
source entity: Pulmonary Disease, Chronic Obstructive
relationship: defined
target entity: Inflammation
sentence: ['(1) Background: Chronic obstructive pulmonary disease (COPD) is defined as an inflammatory disorder that presents an increasingly prevalent health problem.']
source: ['COPD']
target: ['inflammatory disorder']
source type: ['Disease']
target type: ['Disease']
PMID: ['30781849']
DP: ['2019 Feb 13']
date: [20190213]
TI: ['Chronic Obstructive Pulmonary Disease as a Main Factor of Premature Aging.']
TA: ['Int J Environ Res Public Health']
IF: [0.0]
IF5: [0.0]
method: ['deep learning', 'shortest path']


In [8]:
RelationTypes = set()
for k in RelationInfo.keys():
    RelationTypes.add(RelationInfo[k]['relationship'])

print(f"There are {len(RelationInfo)} relations")
print(f"There are {len(RelationTypes)} types of relation")
print(RelationTypes)

There are 116495 relations
There are 3058 types of relation
{'sense', 'convert', 'enlarged', 'work in', 'contribute to', 'carry', 'exit', 'increase formation', 'respond to', 'hybridized', 'point', 'be current mainstay of', 'effect', 'record', 'undertake', 'have make', 'coadminister', 'development of', 'move', 'elaborate', 'focuss', 'score', 'shrunk', 'degenerated', 'recognize', 'regulate metabolism through', 'adher', 'proliferate', 'vaccinate', 'transition', 'Treatment with', 'estimated', 'notice', 'be useful addition to', 'induced', 'incite', 'conclude', 'silencing', 'be prescribe analgesic drug for', 'underline', 'be unique to', 'contrast', 'be strongest predictor of', 'impair', 'OBJECTIVE', 'attributed', 'suffering', 'license', 'less', '-are', 'repress', 'oversee', 'replaced', 'speak', 'challenged', 'tumors', 'supplemented', 'ingest', 'rejuvenate', 'prevenT', 'ameliorate', 'infiltrate', 'be predictors of', 'suffered', 'assessment of', 'normalise', 'differentiated', 'nebulise', 'be i

In [9]:
# How many of each type? Create a dictionary of the types with values set to zero then iterate
RelationCounts = {k: 0 for k in RelationTypes}
for k in RelationInfo.keys():
    RelationCounts[RelationInfo[k]['relationship']] +=1

# Convert dictionary into list of tuples for sorting
RelationCounts = [(k, RelationCounts[k]) for k in RelationCounts.keys()]

from operator import itemgetter
RelationCounts.sort(key=itemgetter(1), reverse=True)
for i in range(5):
    print(RelationCounts[i])

('associated', 19110)
('include', 5542)
('increase', 2088)
('result', 2015)
('cause', 2006)


## 2. Visualisation

Create and visualise the graph and its ontology. You will need to add the package `pyvis` for this. First, let's try out a small examples to see how it works.

In [10]:
from pyvis.network import Network
net = Network()
net.add_node(0,title='a')
net.add_node(1,title='b')
net.add_node(2,title='c')
net.add_node(3,title='d')


net.add_edge(0,1,title='w')
net.add_edge(0,2,title='x')
net.add_edge(1,3,title='y')
net.add_edge(3,0,title='z')

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("nx.html")

In [11]:
# Construct a dictionary of the nodes
nodeindex = dict()
for i, node in enumerate(EntityInfo.keys()):
    nodeindex[node] = (i, f"{EntityInfo[node][0]['type']}: {EntityInfo[node][0]['entity']}")

edgeindex = list()
for k in RelationInfo.keys():
    source = nodeindex[RelationInfo[k]['source entity']][0]
    target = nodeindex[RelationInfo[k]['target entity']][0]
    edgeindex.append((source, target, k))

# the full graph is too big so randomly sample
# flip the node lookup table 
nodelookup = {nodeindex[k][0]: nodeindex[k][1] for k in nodeindex.keys()}
# Select 1000 random edges
import random
randomedges = random.choices(edgeindex,k=5000)
# Get the nodes used in these edged

randomnodes = []
for k in randomedges:
    randomnodes.append(k[0])
    randomnodes.append(k[1])
    
net = Network()
for k in randomnodes:
    net.add_node(k, title=nodelookup[k])

for k in randomedges:
    net.add_edge(k[0],k[1],title=k[2])

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("nx.html")

## 3. Load in the graph from RDF

In [16]:
from owlready2 import *
onto = get_ontology("HALD.rdf").load()

# 4. Querying the graph with SPARQL

Use SPARQL to recompute some basis statistics

* How many types of entity are there and what are their types?
* How many instances of each type of entity are there?
* How many different types of relation are there?
* What are the five most common relations and how many times is each one present?

Let's also try some more 

* Write a SPARQL Query to find all of the relationships between `carbohydrate`  and `mutation`
* Write a SPARQL query to identify all relations that are outgoing from a named node (choose one)
* Write a SPARQL query to identify all relations that are incoming to a named node (choose one)

Here is a simple example to get you started. Note the namespaces.

In [ ]:
x = list(default_world.sparql(
    """
    PREFIX owl: <http://www.w3.org/2002/07/owl#>

    SELECT DISTINCT ?entity 
    WHERE
    {
        ?entity rdf:type owl:Class 
    }
    """))

print(x)